# COMP5318 Assignment 1: Rice Classification

##### Group number: ...
##### Student 1 SID: STUDENT_ID_REDACTED
##### Student 2 SID: ...  
##### Student 3 SID: ... 
##### Student 4 SID: STUDENT_ID_REDACTED

## **1. Data Pre-processing**

In [1]:
# Import all libraries
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Change only this path to process another CSV with the same label format.
filename = "clean_dataset/rice-final2.csv"


In [4]:
def preprocess_data(filename):
    data = pd.read_csv(filename, na_values="?")

    # The last column is the label; all other columns are features.
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1].map({"class1": 0, "class2": 1}).to_numpy()

    # Fill missing values with column means, then scale each feature to [0, 1].
    X = SimpleImputer(strategy="mean").fit_transform(X)
    X = MinMaxScaler().fit_transform(X)
    return X, y


X, y = preprocess_data(filename)


In [5]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(min(n_rows, len(X))):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            

print_data(X, y)


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [6]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [7]:
# Logistic Regression
logreg = LogisticRegression(solver="liblinear", random_state=0)
logreg_scores = cross_val_score(logreg, X, y, cv=cvKFold, scoring="accuracy")


In [8]:
# Naïve Bayes
naive_bayes = GaussianNB()
nb_scores = cross_val_score(naive_bayes, X, y, cv=cvKFold, scoring="accuracy")


### Part 1 Results


In [9]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {logreg_scores.mean():.4f}")
print(f"NB average cross-validation accuracy: {nb_scores.mean():.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [10]:
# KNN 
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]


In [11]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

In [12]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [13]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [14]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to 'sqrt'.
# parameters you may consider
rf_n_estimators = [10, 30, 60, 100]
rf_max_leaf_nodes = [6, 12]

random_forest = RandomForestClassifier(
    criterion="entropy",
    max_features="sqrt",
    random_state=0
)

rf_param_grid = {
    "n_estimators": rf_n_estimators,
    "max_leaf_nodes": rf_max_leaf_nodes
}

In [15]:
# SVM
# parameters you may consider
svm_C = [0.01, 0.1, 1, 5]
svm_gamma = [0.01, 0.1, 1, 10]
# optional
svm_kernel = ["rbf"]

support_vector_machine = SVC(random_state=0)

svm_param_grid = {
    "C": svm_C,
    "gamma": svm_gamma,
    "kernel": svm_kernel
}

### Part 2: Results

In [16]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn).
# The stratified folds from cvKFold should be provided to GridSearchCV.

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0.
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", "n_estimators" and "max_leaf_nodes".

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)

# Random Forest
rf_grid = GridSearchCV(
    estimator=random_forest,
    param_grid=rf_param_grid,
    cv=cvKFold,
    scoring="accuracy"
)
rf_grid.fit(X_train, y_train)
rf_test_predictions = rf_grid.best_estimator_.predict(X_test)

print("RF best n_estimators: {}".format(rf_grid.best_params_["n_estimators"]))
print("RF best max_leaf_nodes: {}".format(rf_grid.best_params_["max_leaf_nodes"]))
print("RF cross-validation accuracy: {:.4f}".format(rf_grid.best_score_))
print("RF test set accuracy: {:.4f}".format(accuracy_score(y_test, rf_test_predictions)))
print("RF test set macro average F1: {:.4f}".format(f1_score(y_test, rf_test_predictions, average="macro")))
print("RF test set weighted average F1: {:.4f}".format(f1_score(y_test, rf_test_predictions, average="weighted")))

# SVM
svm_grid = GridSearchCV(
    estimator=support_vector_machine,
    param_grid=svm_param_grid,
    cv=cvKFold,
    scoring="accuracy"
)
svm_grid.fit(X_train, y_train)
svm_test_predictions = svm_grid.best_estimator_.predict(X_test)

print("SVM best C: {}".format(svm_grid.best_params_["C"]))
print("SVM best gamma: {}".format(svm_grid.best_params_["gamma"]))
print("SVM best kernel: {}".format(svm_grid.best_params_["kernel"]))
print("SVM cross-validation accuracy: {:.4f}".format(svm_grid.best_score_))
print("SVM test set accuracy: {:.4f}".format(accuracy_score(y_test, svm_test_predictions)))

RF best n_estimators: 30
RF best max_leaf_nodes: 6
RF cross-validation accuracy: 0.9411
RF test set accuracy: 0.9429
RF test set macro average F1: 0.9414
RF test set weighted average F1: 0.9427


SVM best C: 5
SVM best gamma: 1
SVM best kernel: rbf
SVM cross-validation accuracy: 0.9429
SVM test set accuracy: 0.9321


### Test your code

In [17]:
#load the test dataset to test out your model 


## **3. Reflection and Discussion**



## **AI Acknowledgement**